# World Cup Predictor

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt

## Elo Team Ratings

In [ ]:
elo_team_ratings = pd.read_csv('elo_wc2026.csv')
elo_team_ratings['country'] = elo_team_ratings['country'].replace({'Czechia': 'Czech Republic'})
elo_team_ratings_2022 = elo_team_ratings[elo_team_ratings['snapshot_date']=='2021-12-31']
elo_team_ratings_2024 = elo_team_ratings[elo_team_ratings['snapshot_date']=='2023-12-31']
elo_team_ratings_2026 = elo_team_ratings[elo_team_ratings['snapshot_date']=='2026-05-27']
elo_team_ratings_2026

In [ ]:
elo_team_ratings_2022['win_rate'] = elo_team_ratings_2022['wins']/elo_team_ratings_2022['matches_total']
elo_team_ratings_2022['loss_rate'] = elo_team_ratings_2022['losses']/elo_team_ratings_2022['matches_total']
elo_team_ratings_2022['draw_rate'] = elo_team_ratings_2022['draws']/elo_team_ratings_2022['matches_total']

elo_team_ratings_2024['win_rate'] = elo_team_ratings_2024['wins']/elo_team_ratings_2024['matches_total']
elo_team_ratings_2024['loss_rate'] = elo_team_ratings_2024['losses']/elo_team_ratings_2024['matches_total']
elo_team_ratings_2024['draw_rate'] = elo_team_ratings_2024['draws']/elo_team_ratings_2024['matches_total']

elo_team_ratings_2026['win_rate'] = elo_team_ratings_2026['wins']/elo_team_ratings_2026['matches_total']
elo_team_ratings_2026['loss_rate'] = elo_team_ratings_2026['losses']/elo_team_ratings_2026['matches_total']
elo_team_ratings_2026['draw_rate'] = elo_team_ratings_2026['draws']/elo_team_ratings_2026['matches_total']

## International Fixtures - get team baseline for predicting WC results. Can train on 2022 and test on 2026?

In [ ]:
internationals = pd.read_csv('international_results.csv')

In [ ]:
internationals = internationals[internationals['date']>='2020-11-20'] # two years of data pre wc 2022

internationals_home = internationals.drop(["away_team"], axis=1)
internationals_home['home_away'] = 'home'
internationals_home = internationals_home.rename(columns={'home_team':'team', 'home_score':'goals_scored', 'away_score':'goals_conceded'})

internationals_away = internationals.drop(["home_team"], axis=1)
internationals_away['home_away'] = 'away'
internationals_away = internationals_away.rename(columns={'away_team':'team', 'away_score':'goals_scored', 'home_score':'goals_conceded'})

internationals_final = pd.concat([internationals_home, internationals_away]).sort_values(by='date').reset_index()
internationals_final

In [ ]:
# remove future games with no results (and remove any current world cup scores given the time the dataset was pulled)
# internationals_final = internationals_final[(internationals_final['date']<'2026-06-11') & (~internationals_final['team'].isna())]

In [ ]:
conditions = [(internationals_final['goals_scored'] > internationals_final['goals_conceded']), 
              (internationals_final['goals_scored'] < internationals_final['goals_conceded']), 
              (internationals_final['goals_scored'] == internationals_final['goals_conceded'])]
choices = ["win", 'loss', 'draw']
    
internationals_final["result"] = np.select(conditions, choices, default=np.nan)

In [ ]:
internationals_final['home_away'] = np.where(internationals_final['neutral'] == True, 'neutral', internationals_final['home_away'])

In [ ]:
internationals_final["goals_scored_rolling_10_games"] = internationals_final.groupby('team').goals_scored.apply(lambda x: x.rolling(10).sum()).reset_index(level=0, drop=True)
internationals_final["goals_conceded_rolling_10_games"] = internationals_final.groupby('team').goals_conceded.apply(lambda x: x.rolling(10).sum()).reset_index(level=0, drop=True)
internationals_final["goals_scored_rolling_20_games"] = internationals_final.groupby('team').goals_scored.apply(lambda x: x.rolling(20).sum()).reset_index(level=0, drop=True)
internationals_final["goals_conceded_rolling_20_games"] = internationals_final.groupby('team').goals_conceded.apply(lambda x: x.rolling(20).sum()).reset_index(level=0, drop=True)
internationals_final["goals_scored_rolling_50_games"] = internationals_final.groupby('team').goals_scored.apply(lambda x: x.rolling(50).sum()).reset_index(level=0, drop=True)
internationals_final["goals_conceded_rolling_50_games"] = internationals_final.groupby('team').goals_conceded.apply(lambda x: x.rolling(50).sum()).reset_index(level=0, drop=True)

In [ ]:
internationals_2022 = internationals_final[internationals_final['date']<'2022-11-20']

internationals_2022['win'] = internationals_2022['result']=='win'

win_rate_2022 = internationals_2022.groupby('team').agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_2022['total_win_rate'] = win_rate_2022['Total_Wins']/win_rate_2022['Total_Games']

win_rate_home_away_2022 = internationals_2022.groupby(['team','home_away']).agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_home_away_2022['total_win_rate'] = win_rate_home_away_2022['Total_Wins']/win_rate_home_away_2022['Total_Games']

pivoted_df = win_rate_home_away_2022.pivot(index='team', columns='home_away', values='total_win_rate').reset_index()

pivoted_df.columns.name = None
pivoted_df.rename(columns={'home': 'home_win_rate', 'away': 'away_win_rate', 'neutral': 'neutral_win_rate'}, inplace=True)

win_rates_2022 = pd.merge(win_rate_2022,pivoted_df, on='team', how='left')


# replicate for 2024
internationals_2024 = internationals_final[(internationals_final['date']>'2022-06-14') & (internationals_final['date']<'2024-07-14') ] #EUROS dates +2 years before

internationals_2024['win'] = internationals_2024['result']=='win'

win_rate_2024 = internationals_2024.groupby('team').agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_2024['total_win_rate'] = win_rate_2024['Total_Wins']/win_rate_2024['Total_Games']

win_rate_home_away_2024 = internationals_2024.groupby(['team','home_away']).agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_home_away_2024['total_win_rate'] = win_rate_home_away_2024['Total_Wins']/win_rate_home_away_2024['Total_Games']

pivoted_df = win_rate_home_away_2024.pivot(index='team', columns='home_away', values='total_win_rate').reset_index()

pivoted_df.columns.name = None
pivoted_df.rename(columns={'home': 'home_win_rate', 'away': 'away_win_rate', 'neutral': 'neutral_win_rate'}, inplace=True)

win_rates_2024 = pd.merge(win_rate_2024,pivoted_df, on='team', how='left')


# replicate for 2026
internationals_2026 = internationals_final[internationals_final['date']>'2024-06-11'] #world cup fixtures only

internationals_2026['win'] = internationals_2026['result']=='win'

win_rate_2026 = internationals_2026.groupby('team').agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_2026['total_win_rate'] = win_rate_2026['Total_Wins']/win_rate_2026['Total_Games']

win_rate_home_away_2026 = internationals_2026.groupby(['team','home_away']).agg(
    Total_Games=('win', 'count'),
    Total_Wins=('win', 'sum')
).reset_index()

win_rate_home_away_2026['total_win_rate'] = win_rate_home_away_2026['Total_Wins']/win_rate_home_away_2026['Total_Games']

pivoted_df = win_rate_home_away_2026.pivot(index='team', columns='home_away', values='total_win_rate').reset_index()

pivoted_df.columns.name = None
pivoted_df.rename(columns={'home': 'home_win_rate', 'away': 'away_win_rate', 'neutral': 'neutral_win_rate'}, inplace=True)

win_rates_2026 = pd.merge(win_rate_2026,pivoted_df, on='team', how='left')

In [ ]:
win_rates_2022 = win_rates_2022.fillna(0)
win_rates_2024 = win_rates_2024.fillna(0)
win_rates_2026 = win_rates_2026.fillna(0)

In [ ]:
most_recent_goals_scored_2022 = internationals_2022.groupby('team')['goals_scored_rolling_10_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2022 = internationals_2022.groupby('team')['goals_conceded_rolling_10_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2024 = internationals_2024.groupby('team')['goals_scored_rolling_10_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2024 = internationals_2024.groupby('team')['goals_conceded_rolling_10_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2026 = internationals_2026.groupby('team')['goals_scored_rolling_10_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2026 = internationals_2026.groupby('team')['goals_conceded_rolling_10_games'].last().reset_index().fillna(0)

most_recent_goals_scored_2022 = internationals_2022.groupby('team')['goals_scored_rolling_20_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2022 = internationals_2022.groupby('team')['goals_conceded_rolling_20_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2024 = internationals_2024.groupby('team')['goals_scored_rolling_20_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2024 = internationals_2024.groupby('team')['goals_conceded_rolling_20_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2026 = internationals_2026.groupby('team')['goals_scored_rolling_20_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2026 = internationals_2026.groupby('team')['goals_conceded_rolling_20_games'].last().reset_index().fillna(0)

most_recent_goals_scored_2022 = internationals_2022.groupby('team')['goals_scored_rolling_50_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2022 = internationals_2022.groupby('team')['goals_conceded_rolling_50_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2024 = internationals_2024.groupby('team')['goals_scored_rolling_50_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2024 = internationals_2024.groupby('team')['goals_conceded_rolling_50_games'].last().reset_index().fillna(0)
most_recent_goals_scored_2026 = internationals_2026.groupby('team')['goals_scored_rolling_50_games'].last().reset_index().fillna(0)
most_recent_goals_conceded_2026 = internationals_2026.groupby('team')['goals_conceded_rolling_50_games'].last().reset_index().fillna(0)

In [ ]:
internationals_2022["goals_scored_per_game_avg"] = internationals_2022.groupby('team')['goals_scored'].expanding().mean().droplevel(0)
internationals_2022["goals_conceded_per_game_avg"] = internationals_2022.groupby('team')['goals_conceded'].expanding().mean().droplevel(0)
internationals_2024["goals_scored_per_game_avg"] = internationals_2024.groupby('team')['goals_scored'].expanding().mean().droplevel(0)
internationals_2024["goals_conceded_per_game_avg"] = internationals_2024.groupby('team')['goals_conceded'].expanding().mean().droplevel(0)
internationals_2026["goals_scored_per_game_avg"] = internationals_2026.groupby('team')['goals_scored'].expanding().mean().droplevel(0)
internationals_2026["goals_conceded_per_game_avg"] = internationals_2026.groupby('team')['goals_conceded'].expanding().mean().droplevel(0)

In [ ]:
international_games_team_summary_2022 = pd.merge(win_rates_2022,pd.merge(most_recent_goals_scored_2022, most_recent_goals_conceded_2022, on='team', how='left'),on='team', how='left')
international_games_team_summary_2024 = pd.merge(win_rates_2024,pd.merge(most_recent_goals_scored_2022, most_recent_goals_conceded_2022, on='team', how='left'),on='team', how='left')
international_games_team_summary_2026 = pd.merge(win_rates_2026,pd.merge(most_recent_goals_scored_2026, most_recent_goals_conceded_2026, on='team', how='left'),on='team', how='left') 

## Merged final dataset

In [ ]:
df_2022 = pd.merge(international_games_team_summary_2022, elo_team_ratings_2022, left_on='team', right_on='country', how='left')

In [ ]:
df_2026 = pd.merge(international_games_team_summary_2026, elo_team_ratings_2026, left_on='team', right_on='country', how='left')

In [ ]:
df_2022.head()

In [ ]:
groups_2026 = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

groups_2022 = {
    "A": ["Qatar", "Ecuador", "Senegal", "Netherlands"],
    "B": ["England", "Iran", "United States", "Wales"],
    "C": ["Argentina", "Saudi Arabia", "Mexico", "Poland"],
    "D": ["France", "Australia", "Denmark", "Tunisia"],
    "E": ["Spain", "Costa Rica", "Germany", "Japan"],
    "F": ["Belgium", "Canada", "Morocco", "Croatia"],
    "G": ["Brazil", "Serbia", "Switzerland", "Cameroon"],
    "H": ["Portugal", "Ghana", "Uruguay", "South Korea"]
}

In [ ]:
team_to_group = {team: group for group, teams in groups_2022.items() for team in teams}
df_2022['Group'] = df_2022['country'].map(team_to_group)
group_mean_difficulty = df_2022.groupby('Group')['rating'].mean().reset_index()
group_mean_difficulty = group_mean_difficulty.rename(columns={'rating':'group_mean_rating'})
data_2022 = pd.merge(df_2022, group_mean_difficulty, on='Group', how='left')

team_to_group = {team: group for group, teams in groups_2026.items() for team in teams}
df_2026['Group'] = df_2026['country'].map(team_to_group)
group_mean_difficulty = df_2022.groupby('Group')['rating'].mean().reset_index()
group_mean_difficulty = group_mean_difficulty.rename(columns={'rating':'group_mean_rating'})
data_2026 = pd.merge(df_2026, group_mean_difficulty, on='Group', how='left')

## Ranking methodology

In [ ]:
pip install catboost lightgbm

In [ ]:
data_final_2022 = pd.merge(internationals_2022, elo_team_ratings_2022, left_on='team', right_on='country')
data_final_2022['opponent_rank'] = data_final_2022.groupby('index')['rank'].shift(1).fillna(data_final_2022.groupby('index')['rank'].shift(-1))
data_final_2024 = pd.merge(internationals_2024, elo_team_ratings_2024, left_on='team', right_on='country')
data_final_2024['opponent_rank'] = data_final_2024.groupby('index')['rank'].shift(1).fillna(data_final_2024.groupby('index')['rank'].shift(-1))
data_final_2026 = pd.merge(internationals_2026, elo_team_ratings_2026, left_on='team', right_on='country')
data_final_2026['opponent_rank'] = data_final_2026.groupby('index')['rank'].shift(1).fillna(data_final_2026.groupby('index')['rank'].shift(-1))

# adding target for training
conditions = [
    data_final_2022['goals_scored'] > data_final_2022['goals_conceded'], #condition 1
    data_final_2022['goals_scored'] < data_final_2022['goals_conceded'], #condition 2
    data_final_2022['goals_scored'] == data_final_2022['goals_conceded'], #condition 3
]

outcomes = [3,0,1]

data_final_2022['target'] = np.select(conditions, outcomes)

### CatBoostRanker

In [ ]:
from catboost import CatBoostRanker, Pool
import lightgbm as lgb

X = data_final_2022.drop(columns = ['goals_scored','goals_conceded','target']).select_dtypes(include=np.number)
y = data_final_2022['target']
data_final_2022_sorted = data_final_2022.sort_values(by='index')
group_id = data_final_2022_sorted['index']

train_pool = Pool(data=X, label=y, group_id=group_id)

model = CatBoostRanker(
    iterations=700,
    depth=4,
    learning_rate=0.54,
    loss_function='YetiRank',
    eval_metric='FilteredDCG',
    # early_stopping_rounds=100,
    random_seed=42
)

model.fit(train_pool)

data_final_2022_sorted['cat_boost_rank_score'] = model.predict(X)

data_final_2022_sorted

In [ ]:
euros = data_final_2024[data_final_2024['tournament'] == 'UEFA Euro']
X_test_euros = euros.drop(columns = ['goals_scored','goals_conceded']).select_dtypes(include=np.number)
X_test_euros
euros['rank_score'] = model.predict(X_test_euros)

In [ ]:
euros_grouped = pd.DataFrame(euros.groupby('team')['rank_score'].mean())

In [ ]:
euros_grouped.sort_values(by='rank_score', ascending=False)

In [ ]:
wc_2026 = data_final_2026[data_final_2026['tournament'] == 'FIFA World Cup']
X_test_wc = wc_2026.drop(columns=['goals_scored','goals_conceded']).select_dtypes(include=np.number)
wc_2026['rank_score'] = model.predict(X_test_wc)

In [ ]:
wc_2026 = pd.DataFrame(wc_2026.groupby('team')['rank_score'].mean().reset_index())
wc_2026 = wc_2026.sort_values(by='rank_score', ascending=False).reset_index(drop=True)

In [ ]:
confed_cols = elo_team_ratings[['country','confederation']]
confed = confed_cols.rename(columns={"country": "team"})
wc_2026_data = pd.merge(wc_2026, confed, left_on='team', right_on='team')
wc_2026_data = wc_2026_data.drop_duplicates().reset_index(drop=True)

In [ ]:
wc_2026_data.to_csv("wc_2026_predictions.csv")

## Post world cup validation

In [ ]:
wc_2026_validation = wc_2026_data.rename_axis('predicted_rank').reset_index()

In [ ]:
actual_placing = {
    "Spain": 1,
    "Argentina": 2,
    "England": 3,
    "France": 4,
    "Belgium": 5,
    "Switzerland": 6,
    "Norway": 7,
    "Morocco": 8,
    "Colombia": 9,
    "Egypt": 10,
    "Portugal": 11,
    "Mexico": 12,
    "Brazil": 13,
    "USA": 14,
    "Canada": 15,
    "Paraguay": 16,
    "Australia": 17,
    "Ghana": 18,
    "Cabo Verde": 19,
    "Austria": 20,
    "Senegal": 21,
    "Japan": 22,
    "Sweden": 23,
    "Ivory Coast":24,
    "Algeria": 25,
    "DR Congo": 26,
    "Bosnia and Herzegovina": 27,
    "Germany": 28,
    "South Africa": 29,
    "Croatia": 30,
    "Ecuador": 31,
    "Netherlands": 32
    
}


wc_2026_validation['actual_rank'] = wc_2026_validation['team'].map(actual_placing)

In [ ]:
wc_2026_validation = wc_2026_validation.fillna(33)
from scipy.stats import spearmanr
rho, p_value = spearmanr(wc_2026_validation['actual_rank'], wc_2026_validation['predicted_rank'])

print(f"Spearman's Rho: {rho:.4f}")
print(f"P-value: {p_value:.4f}")